In [1]:
import cv2

In [2]:
test_img = cv2.imread('../angio-6/test_img.png', cv2.IMREAD_GRAYSCALE)
cv2.imshow('original test image', test_img)
cv2.waitKey(0)

-1

In [3]:
test_mask = cv2.imread('../angio-6/test_img_mask.png', cv2.IMREAD_GRAYSCALE)
cv2.imshow('mask test image', test_mask)
cv2.waitKey(0)

-1

In [4]:
edges = cv2.Canny(test_mask,127,255)
cv2.imshow('test contours', edges)
cv2.waitKey(0)

-1

In [5]:
import numpy as np

np.unique(edges)

array([  0, 255], dtype=uint8)

In [6]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join('..')))

Unet + Canny

In [11]:
from model import Unet
import torch

model = Unet()
model.load_state_dict(torch.load("../resources/model.pth"))
device = 'cuda'
model.to(device)
next(model.parameters()).is_cuda

C:\Users\abram\AppData\Local\Temp\ipykernel_18924\2769380017.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("../resources/model.pth"))


True

FACM

In [12]:
from torchvision.transforms import v2
from time import time
 
def get_predictions(img):
    tensor = v2.ToTensor()(img).unsqueeze(0).to(device)
    out = model(tensor)
    mask = out[0].cpu().detach().permute(1,2,0)
    probs = mask.numpy()
    norm = (mask.numpy()*255).astype(np.uint8)
    labels = cv2.Canny(norm, 127, 255)
    return labels, probs

start = time()
labels, probs = get_predictions(test_img)
print(f'Execution time on {device}: {time()-start}')

c:\soft\miniconda\envs\VCD\lib\site-packages\torchvision\transforms\v2\_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


Execution time on cuda: 0.9383015632629395


In [9]:
np.unique(probs), np.unique(labels)

(array([4.6256732e-06, 4.7247213e-06, 4.7789790e-06, ..., 9.7957981e-01,
        9.7973102e-01, 9.7983563e-01], dtype=float32),
 array([  0, 255], dtype=uint8))

In [16]:
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score

def compute_metircs(ground_true, labels, probs):
    ground_true = ground_true.astype(np.float32)
    labels = labels.astype(np.float32)
    ground_true /= 255
    labels /= 255

    y_true = ground_true.flatten()
    y_pred = labels.flatten()
    y_prob = probs.flatten()

    print(f'F1 score: {f1_score(y_true, y_pred, average="macro")}')
    print(f'Precision score: {precision_score(y_true, y_pred, average="macro")}')
    print(f'Recall score: {recall_score(y_true, y_pred, average="macro")}')
    print(f'ROC AUC score: {roc_auc_score(y_true, y_prob)}')

compute_metircs(edges, labels, probs)

F1 score: 0.6764286372593337
Precision score: 0.6566068195089508
Recall score: 0.7021887570362861
ROC AUC score: 0.9585411157731766
